In [5]:
!sudo apt-get update -qq
!sudo env DEBIAN_FRONTEND=noninteractive apt-get install -y mysql-server
!sudo service mysql start

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
mysql-server is already the newest version (8.0.46-0ubuntu0.24.04.5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
[ OK ]


In [6]:
from google.colab import files
files.upload()   # selecione projeto-integracao-bd-cin-main.zip
!unzip -q projeto-integracao-bd-cin-main.zip
%cd projeto-integracao-bd-cin-main

Saving projeto-integracao-bd-cin-main.zip to projeto-integracao-bd-cin-main.zip
/content/projeto-integracao-bd-cin/projeto-integracao-bd-cin-main/projeto-integracao-bd-cin-main


In [7]:
%pip install "pandas>=2.2,<4" "mysql-connector-python>=9,<10"

In [8]:
!sudo mysql -e "CREATE DATABASE IF NOT EXISTS zika_dw CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_bin; CREATE USER IF NOT EXISTS 'zika'@'localhost' IDENTIFIED BY 'zika123'; GRANT ALL ON zika_dw.* TO 'zika'@'localhost';"

import os
os.environ.update({"MYSQL_HOST":"127.0.0.1","MYSQL_PORT":"3306",
                   "MYSQL_USER":"zika","MYSQL_PASSWORD":"zika123","MYSQL_DATABASE":"zika_dw"})

In [9]:
import sys; sys.path.insert(0, ".")
from pipelines.core import extract, transform, star
raw = extract()
print(f"Linhas brutas: {len(raw):,}")
rows, rejected, audit = transform(raw)
tables = star(rows)
print(audit)
from pipelines.database import connect, publish, validate
with connect() as connection:
    publish(connection, tables)
    counts = validate(connection)
print(counts)

Linhas brutas: 82,460
{'brutas': 82460, 'duplicadas_exatas': 56, 'rejeitadas': 0, 'validas': 82404}
{'dim_tempo': 1003, 'dim_localidade': 2562, 'dim_unidade_notificadora': 6631, 'dim_paciente': 9024, 'dim_investigacao_desfecho': 128, 'fato_notificacao_zika': 82404}


In [10]:
from pipelines.run import run
manifesto = run("etl")
print(manifesto)

{
  "pipeline": "etl",
  "reconciliacao": {
    "brutas": 82460,
    "duplicadas_exatas": 56,
    "rejeitadas": 0,
    "validas": 82404
  },
  "tabelas": {
    "dim_tempo": 1003,
    "dim_localidade": 2562,
    "dim_unidade_notificadora": 6631,
    "dim_paciente": 9024,
    "dim_investigacao_desfecho": 128,
    "fato_notificacao_zika": 82404
  },
  "duracao_segundos": 31.061
}
{'pipeline': 'etl', 'reconciliacao': {'brutas': 82460, 'duplicadas_exatas': 56, 'rejeitadas': 0, 'validas': 82404}, 'tabelas': {'dim_tempo': 1003, 'dim_localidade': 2562, 'dim_unidade_notificadora': 6631, 'dim_paciente': 9024, 'dim_investigacao_desfecho': 128, 'fato_notificacao_zika': 82404}, 'duracao_segundos': 31.061}
